# 02 - RAG Generation with HF Wikipedia

This notebook runs headline-only RAG using the embedded Wikipedia dataset from Hugging Face (`not-lain/wikipedia`, revision `embedded`). It evaluates Llama, Qwen, and Mistral with Wikipedia corpus sizes from 5k to 25k documents. Word-pair examples are generated without retrieved context.

## Shared settings

The complete matrix contains 15 runs: three models multiplied by five Wikipedia corpus sizes. `K` is the number of retrieved contexts passed to the generation model. Existing outputs are skipped by default, so an interrupted batch can be resumed.

In [ ]:
from pathlib import Path
import subprocess
import sys

MODELS = ("llama", "qwen", "mistral")
N_DOC_COUNTS = tuple(range(5_000, 25_001, 5_000))
DATA = "data/raw/task-a-en.tsv"
RAG_CONFIG = "configs/rag.yaml"
K = 4
APPLY_TO = "headline"
SKIP_EXISTING = True
LOCAL_OUT = Path("data/generated/rag")

def run_rag_matrix(output_dir, output_suffix=""):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    total = len(MODELS) * len(N_DOC_COUNTS)
    completed = 0

    for model in MODELS:
        for n_docs in N_DOC_COUNTS:
            output_path = output_dir / (
                f"{model}_rag_wiki_headline_"
                f"{n_docs // 1000}k_k{K}{output_suffix}.jsonl"
            )
            if SKIP_EXISTING and output_path.exists():
                completed += 1
                print(f"[{completed}/{total}] Skipping existing output: {output_path}")
                continue

            output_path.parent.mkdir(parents=True, exist_ok=True)
            cmd = [
                sys.executable, "scripts/run_rag_generate.py",
                "--model", model,
                "--input", DATA,
                "--output", str(output_path),
                "--rag-config", RAG_CONFIG,
                "--n-docs", str(n_docs),
                "--k", str(K),
                "--apply-to", APPLY_TO,
                "--overwrite",
            ]
            print(f"[{completed + 1}/{total}] model={model}, documents={n_docs:,}")
            subprocess.run(cmd, check=True)
            completed += 1
            print(f"Saved: {output_path}")

    print(f"Matrix complete: {completed}/{total} outputs available.")

print(f"Models: {', '.join(MODELS)}")
print(f"Wikipedia corpus sizes: {[f'{n // 1000}k' for n in N_DOC_COUNTS]}")
print(f"Runs per environment: {len(MODELS) * len(N_DOC_COUNTS)}")

# Remote execution on Colab

Run this chapter in Google Colab. The repository is expected at `/content/LLM-Project-SemEval-Humor-Generation`. Google Drive is mounted and RAG outputs are stored permanently in `MyDrive/semeval_humor_outputs/rag`. Llama also requires an accepted Hugging Face license and an authenticated Hugging Face account.

In [ ]:
%cd /content/LLM-Project-SemEval-Humor-Generation
%pip install -r requirements-colab.txt

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

COLAB_OUT = Path("/content/drive/MyDrive/semeval_humor_outputs/rag")
COLAB_OUT.mkdir(parents=True, exist_ok=True)
print(f"Colab outputs: {COLAB_OUT}")

## Run the complete Colab matrix

Set `RUN_COLAB_MATRIX` to `True` to launch all 15 combinations. Every JSONL is written directly to `COLAB_OUT` on Google Drive, so completed runs survive a Colab disconnection.

In [ ]:
RUN_COLAB_MATRIX = False

if RUN_COLAB_MATRIX:
    run_rag_matrix(COLAB_OUT)
else:
    print("Set RUN_COLAB_MATRIX = True to launch all 15 Colab runs.")

## Check Colab outputs

In [ ]:
for model in MODELS:
    for n_docs in N_DOC_COUNTS:
        output_path = COLAB_OUT / (
            f"{model}_rag_wiki_headline_{n_docs // 1000}k_k{K}.jsonl"
        )
        status = "ready" if output_path.exists() else "missing"
        print(f"{status:7}  {output_path}")

# Local execution with the .venv kernel

Run this chapter on a local workstation after selecting the project `.venv` as the Jupyter kernel. The subprocesses use `sys.executable`, so generation runs with that same environment.

In [ ]:
ROOT = Path.cwd()
if not (ROOT / "scripts" / "run_rag_generate.py").exists():
    raise RuntimeError(f"Run this notebook from the repository root, current path: {ROOT}")
print(f"Repository root: {ROOT}")
print(f"Python interpreter: {sys.executable}")

## Local dependency install

Run this once in the selected `.venv` kernel. It can be skipped when the environment already contains the project dependencies and GPU stack.

In [ ]:
%pip install -r requirements-colab.txt

## Run the complete local matrix

Set `RUN_LOCAL_MATRIX` to `True` to launch all 15 combinations. Local output names use `_local` to avoid overwriting Colab results.

In [ ]:
RUN_LOCAL_MATRIX = False

if RUN_LOCAL_MATRIX:
    run_rag_matrix(LOCAL_OUT, output_suffix="_local")
else:
    print("Set RUN_LOCAL_MATRIX = True to launch all 15 local runs.")

## Check local outputs

In [ ]:
for model in MODELS:
    for n_docs in N_DOC_COUNTS:
        output_path = LOCAL_OUT / (
            f"{model}_rag_wiki_headline_{n_docs // 1000}k_k{K}_local.jsonl"
        )
        status = "ready" if output_path.exists() else "missing"
        print(f"{status:7}  {output_path}")